In [0]:
# ----------------------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 6.1_ingestao_presencas_votacoes
# LOCAL: 1_bronze/src/6_monitor_presenca/
# OBJETIVO: Simular votações de deputados para obter tendências de alinhamento entre deputados (mesma frente). Usa como fonte a gold_atlas_frentes_parlamentares e fornece a bronze_presenca_votacoes para ser usada no script 3_gold/../6.1_analise_absentismo
# ENTREGA: Desenvolver uma visão sobre junção entre eventos e votações para medir ausências em votações
# ----------------------------------------------------------------------------------------------------------------------------------------------------


from pyspark.sql import Row
import random

# Criação de dados simulados baseados na lista real de deputados
 
df_deputados = spark.table("gold_atlas_frentes_parlamentares") \
    .select("id_deputado", "nome_deputado").distinct().collect()

votos_mock = []
# Criados 30 IDs de votação para se ter volume (10 por tema)
ids_votacoes = list(range(999001, 999031))

print("Gerando 30 votações simuladas para 100 deputados...")

for id_v in ids_votacoes:
    # Definida uma 'probabilidade de falta' para esta votação específica
    prob_presenca = random.uniform(0.4, 0.95) 
    
    for dep in df_deputados:
        if random.random() < prob_presenca:
            votos_mock.append(Row(
                id_votacao=id_v,
                id_deputado=int(dep['id_deputado']),
                nome_deputado=dep['nome_deputado'],
                voto=random.choice(['Sim', 'Não', 'Abstenção']),
                data_captura="2026-05-10"
            ))

df_votos = spark.createDataFrame(votos_mock)
df_votos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_presenca_votacoes")
print(f"Tabela Bronze de Votos agora tem {len(votos_mock)} registros!")

display(df_votos)